# GatingBN (Dynamic Channel Gating + BN Anchor) Example

## Overview

**GatingBN**은 딥페이크 탐지를 위한 Test-Time Adaptation 방법입니다.

**핵심 아이디어:**
- Channel gating으로 신뢰할 수 없는 채널 억제
- BN Anchor로 source manifold 유지 (분포 drift 방지)

**Architecture:**
```
Base Model (frozen) -> layer4 features (F)
    |
    v
+----------------------------------+
|  1. Dynamic Channel Gating       |
|     1x1 Conv -> sigmoid -> gate  |
|     F' = F * gate                |
|                                  |
|  2. BN Anchor                    |
|     F'' = (F' - mu_src) / std_src|
+----------------------------------+
    |
    v
Classifier (frozen) -> logits
```

**TTA Loss:**
```
L = lambda_bn * L_bn + lambda_ent * L_ent

L_bn = |mean(F') - mu_src| + |std(F') - std_src|
L_ent = -sum(p * log(p))
```

**Key Points:**
- 1x1 Conv 기반 동적 gating: 각 spatial location마다 다른 gate
- BN alignment loss가 main regularizer
- Backbone, Head 고정, Gating만 학습

## 1. Import

In [1]:
import sys
# Clear cache
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['NPR', 'npr', 'LGrad', 'lgrad', 'gating']):
        del sys.modules[mod]

In [2]:
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

# Dataset and metrics
from utils.data.dataset import CorruptedDataset
from utils.eval.metrics import PredictionCollector, MetricsCalculator

# Models
from model.LGrad.lgrad_model import LGrad
from model.NPR.npr_model import NPR

# GatingBN
from model.method.gating_bn import UnifiedGatingBN, GatingBNConfig

In [3]:
!nvidia-smi

Sun Jan 25 15:16:01 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.230.02             Driver Version: 535.230.02   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-16GB           Off | 00000000:04:00.0 Off |                    0 |
| N/A   37C    P0              27W / 250W |      6MiB / 16384MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## 2. Configuration

In [4]:
# Device
DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Model selection
MODEL = "NPR"  # "LGrad" or "NPR"

# Datasets
DATASETS = ["ADM", "DDPM", "IDDPM", "LDM", "PNDM", "VQDIFFUSION", "SDV1", "SDV2", 
            "PROGAN", "STYLEGAN", "STYLEGAN2", "BIGGAN", "CYCLEGAN", "STARGAN", "GAUGAN", "DEEPFAKE"]

# Corruptions and Severities
CORRUPTIONS = ["color_contrast", "color_saturation", "resize", "gaussian_blur"]
SEVERITIES = ["corrupted1", "corrupted2", "corrupted3", "corrupted4", "corrupted5"]

# Paths
DATA_ROOT = "corrupted_dataset"
CLEAN_DATA_ROOT = "dataset"  # Clean data for source statistics

# GatingBN config
TAU = 1.0                    # Temperature for gating
INIT_BIAS = 2.0              # Initial gate bias (sigmoid(2) ~ 0.88)
LAMBDA_BN = 1.0              # BN alignment loss weight (main)
LAMBDA_ENT = 0.1             # Entropy loss weight (auxiliary)
TTA_LR = 1e-3                # Learning rate
MAX_TTA_STEPS = 10           # Max adaptation steps
ENABLE_TTA = True

BATCH_SIZE = 16

Using device: cuda:1


## 3. Load Base Model

In [5]:
if MODEL == "LGrad":
    STYLEGAN_WEIGHTS = "model/LGrad/weights/karras2019stylegan-bedrooms-256x256_discriminator.pth"
    CLASSIFIER_WEIGHTS = "model/LGrad/weights/LGrad-Pretrained-Model/LGrad-4class-Trainon-Progan_car_cat_chair_horse.pth"
    
    base_model = LGrad(
        stylegan_weights=STYLEGAN_WEIGHTS,
        classifier_weights=CLASSIFIER_WEIGHTS,
        device=DEVICE
    )
    
    transform = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
    ])
    
elif MODEL == "NPR":
    NPR_WEIGHTS = "model/NPR/weights/NPR.pth"
    
    base_model = NPR(
        weights=NPR_WEIGHTS,
        device=DEVICE
    )
    
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

print(f"Base model loaded: {MODEL}")

/workspace/robust_deepfake_ai/model/NPR/npr_model.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(weights, map_location="cpu")


Base model loaded: NPR


## 4. Create GatingBN Model

In [7]:
# Create GatingBN config
config = GatingBNConfig(
    model=MODEL,
    # optimizer="SGD",
    target_layer=None,  # Auto-detect
    tau=TAU,
    init_bias=INIT_BIAS,
    lambda_bn=LAMBDA_BN,
    lambda_ent=LAMBDA_ENT,
    tta_lr=TTA_LR,
    max_tta_steps=MAX_TTA_STEPS,
    enable_tta=ENABLE_TTA,
    device=DEVICE,
)

# Create GatingBN model
gating_model = UnifiedGatingBN(base_model, config)

# Print info
print("="*60)
print("GatingBN Model Configuration")
print("="*60)
print(f"Model: {config.model}")
print(f"Target Layer: {config.target_layer}")
print("-"*60)
print("Gating Parameters:")
print(f"  Tau (temperature): {config.tau}")
print(f"  Init Bias: {config.init_bias} (sigmoid({config.init_bias})={torch.sigmoid(torch.tensor(config.init_bias)).item():.3f})")
print("-"*60)
print("Loss Weights:")
print(f"  lambda_bn: {config.lambda_bn} (main)")
print(f"  lambda_ent: {config.lambda_ent} (auxiliary)")
print("-"*60)
print("TTA Parameters:")
print(f"  Learning Rate: {config.tta_lr}")
print(f"  Max Steps: {config.max_tta_steps}")
print(f"  Enable TTA: {config.enable_tta}")
print("="*60)

GatingBN Model Configuration
Model: NPR
Target Layer: model.layer2
------------------------------------------------------------
Gating Parameters:
  Tau (temperature): 1.0
  Init Bias: 2.0 (sigmoid(2.0)=0.881)
------------------------------------------------------------
Loss Weights:
  lambda_bn: 1.0 (main)
  lambda_ent: 0.1 (auxiliary)
------------------------------------------------------------
TTA Parameters:
  Learning Rate: 0.001
  Max Steps: 10
  Enable TTA: True


## 5. Collect Source Statistics

Clean data에서 feature의 mean/std를 수집합니다. 이 통계는 BN Anchor에서 사용됩니다.

In [ ]:
# Option 1: Load from clean dataset
# clean_dataset = ...  # Your clean dataset
# clean_loader = DataLoader(clean_dataset, batch_size=BATCH_SIZE, shuffle=False)
# gating_model.collect_source_stats(clean_loader)

# Option 2: Use corrupted dataset's uncorrupted version as proxy
# For now, we'll compute from first batch of test data as a simple baseline

# Option 3: Manual computation (recommended for production)
# mu_src, std_src = precomputed_values
# gating_model.set_source_stats(mu_src, std_src)

In [8]:
# For this example, we'll use a quick approximation:
# Collect stats from first dataset's first corruption level

dataset = CorruptedDataset(
    root=DATA_ROOT,
    datasets=[DATASETS[0]],  # First dataset only
    corruptions=[CORRUPTIONS[0]],
    severities=["corrupted1"],  # Lightest corruption as proxy
    transform=transform
)

# Take subset for quick stats collection
subset_indices = list(range(min(500, len(dataset))))
subset = Subset(dataset, subset_indices)
source_loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Collecting source statistics from {len(subset)} samples...")
gating_model.collect_source_stats(source_loader)

Source statistics collected from 500 samples
  mu_src: mean=2.2271, std=1.8303
  std_src: mean=3.0824, std=2.1566


## 6. Create Test Dataset

In [9]:
dataset = CorruptedDataset(
    root=DATA_ROOT,
    datasets=DATASETS,
    corruptions=CORRUPTIONS,
    severities=SEVERITIES,
    transform=transform
)

print(f"Total samples: {len(dataset)}")
print(f"Datasets: {DATASETS}")
print(f"Corruptions: {CORRUPTIONS}")
print(f"Severities: {SEVERITIES}")

Total samples: 1852120
Datasets: ['ADM', 'DDPM', 'IDDPM', 'LDM', 'PNDM', 'VQDIFFUSION', 'SDV1', 'SDV2', 'PROGAN', 'STYLEGAN', 'STYLEGAN2', 'BIGGAN', 'CYCLEGAN', 'STARGAN', 'GAUGAN', 'DEEPFAKE']
Corruptions: ['color_contrast', 'color_saturation', 'resize', 'gaussian_blur']
Severities: ['corrupted1', 'corrupted2', 'corrupted3', 'corrupted4', 'corrupted5']


## 7. Evaluation Functions

In [10]:
def evaluate_gating_bn(model, dataloader, device, name="test"):
    """
    Evaluate GatingBN model on entire dataloader.
    """
    model.eval()
    collector = PredictionCollector()
    calc = MetricsCalculator()
    
    pbar = tqdm(dataloader, desc=name)
    for batch in pbar:
        images, labels, metadata = batch
        images = images.to(device)
        
        # # Reset gating before each batch
        # model.reset()
        
        logits = model(images)
        probs = torch.sigmoid(logits).squeeze(1)
        
        collector.update(labels, probs.cpu(), threshold=0.5)
    
    metrics = calc.compute_from_collector(collector, name=name)
    return metrics

In [11]:
def evaluate_base_model(model, dataloader, device, name="base"):
    """
    Evaluate base model without any adaptation.
    """
    model.eval()
    collector = PredictionCollector()
    calc = MetricsCalculator()

    pbar = tqdm(dataloader, desc=name)
    with torch.no_grad():
        for batch in pbar:
            images, labels, metadata = batch
            images = images.to(device)

            logits = model(images)
            if isinstance(logits, (tuple, list)):
                logits = logits[0]
            probs = torch.sigmoid(logits).squeeze(1)
            collector.update(labels, probs.cpu(), threshold=0.5)

    metrics = calc.compute_from_collector(collector, name=name)
    return metrics

In [ ]:
# Evaluate on all combinations
# Order: corruption -> severity (Gradual TTA style)
# e.g., color_contrast/1 -> color_contrast/2 -> ... -> color_contrast/5 -> color_saturation/1 -> ...

calc = MetricsCalculator()
all_results = {}  # {(dataset, corruption, severity): metrics}
corruption_avg_results = {}  # {(dataset, corruption): averaged_metrics}

for dataset_name in DATASETS:
    # Reset NRAM model when switching to a new dataset
    gating_model.reset()
    print(f"\n{'#'*70}")
    print(f"# Dataset: {dataset_name} (NRAM model reset)")
    print(f"{'#'*70}")
    
    for corruption in CORRUPTIONS:
        severity_results = []  # Store results for severity 1-5
        
        print(f"\n{'='*70}")
        print(f"Corruption: {corruption}")
        print(f"{'='*70}")
        
        for severity in SEVERITIES:
            # Get indices for this specific combination
            indices = [
                i for i, s in enumerate(dataset.samples)
                if s['dataset'] == dataset_name 
                and s['corruption'] == corruption 
                and s['severity'] == severity
            ]
            
            if len(indices) == 0:
                print(f"  {severity}: No samples, skipping")
                continue
            
            print(f"\n  [{severity}] Samples: {len(indices)}")
            
            # Create dataloader (shuffle=False to maintain order)
            dataloader = DataLoader(
                Subset(dataset, indices),
                batch_size=BATCH_SIZE,
                shuffle=False,
                num_workers=4,
                drop_last=True
            )
            
            # Evaluate with NRAM (TTA happens automatically if enabled)
            # Note: Model state is maintained across severities (continual adaptation)
            metrics = evaluate_gating_bn(
                model=gating_model,
                dataloader=dataloader,
                device=DEVICE,
                name=f"{corruption}-{severity}"
            )
            
            # Print results for this severity
            print(f"      Acc: {metrics['accuracy']*100:.2f}%  AUC: {metrics['auc']*100:.2f}%  AP: {metrics['ap']*100:.2f}%  F1: {metrics['f1']*100:.2f}%")
            
            # Store results
            all_results[(dataset_name, corruption, severity)] = metrics
            severity_results.append(metrics)
        
        # Compute average across severities 1-5 for this corruption
        if severity_results:
            avg_metrics = {
                'accuracy': np.mean([m['accuracy'] for m in severity_results]),
                'auc': np.mean([m['auc'] for m in severity_results]),
                'ap': np.mean([m['ap'] for m in severity_results]),
                'f1': np.mean([m['f1'] for m in severity_results]),
            }
            corruption_avg_results[(dataset_name, corruption)] = avg_metrics
            
            print(f"\n  >> {corruption} Average (severity 1-5):")
            print(f"      Acc: {avg_metrics['accuracy']*100:.2f}%  AUC: {avg_metrics['auc']*100:.2f}%  AP: {avg_metrics['ap']*100:.2f}%  F1: {avg_metrics['f1']*100:.2f}%")

# ============================================================
# Summary Tables
# ============================================================
print(f"\n\n{'='*80}")
print(f"RESULTS SUMMARY (NRAM - TTA {'Enabled' if ENABLE_TTA else 'Disabled'})")
print(f"{'='*80}")

# Table 1: Detailed results per severity
print(f"\n[Table 1] Detailed Results (per severity)")
print(f"{'Corruption':<20} {'Severity':<12} {'Accuracy':<10} {'AUC':<10} {'AP':<10} {'F1':<10}")
print("-" * 72)
for (dataset_name, corruption, severity), metrics in all_results.items():
    print(f"{corruption:<20} {severity:<12} {metrics['accuracy']*100:>6.2f}%    {metrics['auc']*100:>6.2f}%    {metrics['ap']*100:>6.2f}%    {metrics['f1']*100:>6.2f}%")

# Table 2: Averaged results per corruption (main result)
print(f"\n\n[Table 2] Averaged Results (severity 1-5 mean) - MAIN RESULT")
print(f"{'Corruption':<25} {'Accuracy':<10} {'AUC':<10} {'AP':<10} {'F1':<10}")
print("-" * 65)
for (dataset_name, corruption), metrics in corruption_avg_results.items():
    print(f"{corruption:<25} {metrics['accuracy']*100:>6.2f}%    {metrics['auc']*100:>6.2f}%    {metrics['ap']*100:>6.2f}%    {metrics['f1']*100:>6.2f}%")

# Overall average
overall_avg = {
    'accuracy': np.mean([m['accuracy'] for m in corruption_avg_results.values()]),
    'auc': np.mean([m['auc'] for m in corruption_avg_results.values()]),
    'ap': np.mean([m['ap'] for m in corruption_avg_results.values()]),
    'f1': np.mean([m['f1'] for m in corruption_avg_results.values()]),
}
print("-" * 65)
print(f"{'Overall Average':<25} {overall_avg['accuracy']*100:>6.2f}%    {overall_avg['auc']*100:>6.2f}%    {overall_avg['ap']*100:>6.2f}%    {overall_avg['f1']*100:>6.2f}%")


######################################################################
# Dataset: ADM (NRAM model reset)
######################################################################

Corruption: color_contrast

  [corrupted1] Samples: 7000


color_contrast-corrupted1: 100%|██████████| 437/437 [01:20<00:00,  5.43it/s]


      Acc: 93.24%  AUC: 97.27%  AP: 99.41%  F1: 96.19%

  [corrupted2] Samples: 7000


color_contrast-corrupted2:  22%|██▏       | 94/437 [00:18<01:07,  5.05it/s]

## Summary

### GatingBN Key Features

1. **Dynamic 1x1 Conv Gating**
   - 각 spatial location에서 독립적인 gate 계산
   - GAP와 달리 spatial 정보 보존

2. **BN Anchor**
   - Source statistics로 feature 정규화
   - Gating으로 인한 분포 drift 방지

3. **Loss 구성**
   - L_bn (main): source manifold 유지
   - L_ent (auxiliary): 예측 confidence 향상

4. **학습 대상**
   - Gating의 1x1 Conv만 업데이트
   - Backbone, Head 완전 고정